# 01 · Complete preprocessing run (protocol 1.1)

Sandesh Lamichhane, Assistant Professor, Yenepoya (Deemed to be University).

This one notebook does the whole remaining preprocessing. Run every cell **from top to bottom**, one at a time,
waiting for each to finish. You do not need to add, paste or edit anything.

**Before you start:** *Runtime → Change runtime type → High-RAM* (CPU is fine, no GPU needed).

| Step | What it does | Time |
|---|---|---|
| 1 | connect Drive and GitHub, update the code, run the tests | 2 min |
| 2 | schema profile of all 10 datasets | 5 min |
| 3 | CICIDS2017 | 10 min |
| 4 | CICIoMT2024 | 15 min |
| 5 | ToN_IoT | 30–45 min |
| 6 | CIC-IoT2023 | 45–60 min |
| 7 | CSE-CIC-IDS2018, shared CIC features, cross-dataset overlap | 1.5–3 h |
| 8 | LANL host-hour features and Level 3 split | 1–3 h |
| 9 | summary table | 1 min |

Every step saves its results to Drive and pushes them to GitHub before the next one starts, so nothing is lost
if Colab disconnects. **If it disconnects:** run step 1 again, then continue from the step that did not finish.

**Where results go:** reports, data cards and logs in
`MyDrive/PhD-AI-Cyber/code/phd-ai-threat-detection/{results/data_audit, docs/data_cards}` and the same paths on
GitHub; cleaned data in `MyDrive/PhD-AI-Cyber/data/processed/`, splits in `MyDrive/PhD-AI-Cyber/data/splits/`.

## Step 1 · Connect and update (expected end: the latest commit and `26 passed`)

In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')

import os, pathlib, shutil, subprocess
DRIVE_ROOT = '/content/drive/MyDrive/PhD-AI-Cyber'
REPO_DIR   = f'{DRIVE_ROOT}/code/phd-ai-threat-detection'
os.environ['PHD_DRIVE_ROOT'] = DRIVE_ROOT
os.environ['PHD_SCRATCH'] = '/content/scratch'
TOKEN  = userdata.get('GITHUB_TOKEN')
REMOTE = f'https://{TOKEN}@github.com/sandesh20lamichhane/phd-ai-threat-detection.git'

def git(*args, check=False):
    r = subprocess.run(['git', '-C', REPO_DIR, *args], capture_output=True, text=True)
    out = (r.stdout + r.stderr).replace(TOKEN, '***').strip()
    if out:
        print(out)
    if check and r.returncode != 0:
        raise SystemExit(f'git {args[0]} failed (see message above)')
    return r.returncode

git('config', 'user.name', 'Sandesh Lamichhane')
git('config', 'user.email', 'san30lamichhane@gmail.com')
git('config', 'rebase.autoStash', 'true')

# Notebook copies that were uploaded into the repo folder would block the update: move them to backups/.
untracked = subprocess.run(['git', '-C', REPO_DIR, 'ls-files', '--others', '--exclude-standard', 'notebooks'],
                           capture_output=True, text=True).stdout.split()
for rel in untracked:
    pathlib.Path(DRIVE_ROOT, 'backups').mkdir(exist_ok=True)
    shutil.move(f'{REPO_DIR}/{rel}', f'{DRIVE_ROOT}/backups/{pathlib.Path(rel).name}')
    print('moved to backups/:', rel)

# Colab saves outputs into notebooks it has open; those saved outputs are not records, so reset them
# before updating (the results themselves live in results/ and are never touched here).
git('checkout', '--', 'notebooks')
git('pull', '--rebase', REMOTE, 'main', check=True)
print('commit', subprocess.run(['git', '-C', REPO_DIR, 'log', '--oneline', '-1'], capture_output=True, text=True).stdout)
assert pathlib.Path(REPO_DIR, 'src/data/pipeline.py').exists()
%cd $REPO_DIR
!pip -q install -r requirements.txt
!python -m pytest -q tests

def push(message):
    """Save results from the Drive copy of the repo to GitHub, so both hold the same records."""
    git('add', 'results/data_audit', 'docs/data_cards', 'config/feature_mapping')
    if git('commit', '-m', message) != 0:
        print('nothing new to commit')
    git('pull', '--rebase', REMOTE, 'main', check=True)
    git('push', REMOTE, 'HEAD:main', check=True)
    print('saved to GitHub:', message)

def run(*args, message):
    """Run one preprocessing command (its output shows below); push the results only if it succeeded."""
    ip = get_ipython()
    ip.system('python scripts/preprocess.py ' + ' '.join(args))
    ok = ip.user_ns.get('_exit_code', 1) == 0
    push(message if ok else message + ' (FAILED, log only)')
    if not ok:
        raise SystemExit('This step failed. Copy the error above into the project thread.')


## Step 2 · Schema profile (expected: a table with `none` under problems, except LANL's note)

In [ ]:
run('--profile', message='Schema profile (protocol 1.1)')
from IPython.display import Markdown
Markdown(open('results/data_audit/schema_profile.md').read())

## Step 3 · CICIDS2017

In [ ]:
run('--dataset', 'cicids2017_improved', message='Preprocess CICIDS2017 (protocol 1.1)')

## Step 4 · CICIoMT2024

In [ ]:
run('--dataset', 'ciciomt2024', message='Preprocess CICIoMT2024 (protocol 1.1)')

## Step 5 · ToN_IoT

In [ ]:
run('--dataset', 'ton_iot', message='Preprocess ToN_IoT (protocol 1.1)')

## Step 6 · CIC-IoT2023

In [ ]:
run('--dataset', 'ciciot2023', message='Preprocess CIC-IoT2023 (protocol 1.1)')

## Step 7 · CSE-CIC-IDS2018, then freeze the shared CIC feature list and measure cross-dataset overlap

In [ ]:
run('--dataset', 'csecicids2018_improved', message='Preprocess CSE-CIC-IDS2018 (protocol 1.1)')
run('--shared-features', message='Freeze shared CICIDS2017/CSE-CIC-IDS2018 feature list')
run('--cross-overlap', message='Cross-dataset overlap, CICIDS2017 train vs external targets')

## Step 8 · LANL (resumes per source file if Colab disconnects)

In [ ]:
run('--dataset', 'lanl_cyber1', message='LANL host-hour features, compromises and Level 3 split (protocol 1.1)')

## Step 9 · Summary (expected: one row per processed dataset). Then tell the project thread you are done.

In [ ]:
import json, glob
import pandas as pd
rows = []
for f in sorted(glob.glob('results/data_audit/*/report.json')):
    r = json.load(open(f))
    c = r.get('counts', {})
    rows.append({'dataset': r['dataset'], 'rows read': c.get('rows_read'), 'duplicates removed': c.get('exact_duplicates_removed'),
                 'rows kept': c.get('rows_out', r.get('host_hours')), 'built': r['run']['utc'],
                 'code': (r['run']['git_commit'] or '')[:7], 'pilot?': bool(r.get('max_rows'))})
pd.DataFrame(rows)